In [6]:
from glob import glob
from tqdm import tqdm
from datetime import datetime
import netCDF4 as nc
import numpy as np
import pandas as pd
import os
import datetime as dt
import xarray as xr
import matplotlib.dates as mdate
import glob
from sklearn.linear_model import LinearRegression

In [7]:
# function to obtain correlation during night time measurements
def get_linear_reg_coef(df):
    x = df["aLH676"].values.reshape(-1, 1)
    y = df["chla"].values.reshape(-1, 1)
    model = LinearRegression()
    model.fit(x, y)
    return pd.Series({"gradient": model.coef_.item(), "intercept": model.intercept_.item()})

In [8]:
# Loading netcdf files and applying get_linear_reg_coef function to all measurements

path = r"C:\Projects\datalakes\thetis-vertical-profiler\thetis-vertical-profiler\notebooks\data_test\Level2" #example dataset downloaded from datalakes
files = glob.glob(os.path.join (path, "*.nc"))

df_total = pd.DataFrame()

In [9]:
for file in files:
    print(file)
    # converting netcdf files to dataframes and formatting for easier processing
    data = xr.open_dataset(file)
    df = data.to_dataframe()
    df = df.reset_index()
    df_chl = df[["depth", "time", "par", "chla", "aLH676"]]
    df_chl = df_chl.dropna()
    df_chl = df_chl.drop_duplicates(["time", "par"], keep="first").sort_values(["time","depth"])
    df_chl = df_chl.rename(columns={"time":"datetime"})
    df_chl["date"] = pd.to_datetime(df_chl["datetime"]).dt.date
    df_chl["time"] = pd.to_datetime(df_chl["datetime"]).dt.time
    
    print(df_chl)
    
    # isolating night time measurements using PAR and depth of < 3 m to ensure that the low PAR values at depth during day time are not included
    df_chl_night = df_chl[(df_chl["depth"] < 3) & (df_chl["par"] < 50)]
    df_chl_night = df_chl.merge(df_chl_night, on=["datetime"])
    df_chl_night = df_chl_night.drop(columns=["depth_y","par_y","chla_y","aLH676_y","date_y","time_y"])
    df_chl_night = df_chl_night.rename(columns={"depth_x":"depth","par_x":"par","chla_x":"chla","aLH676_x":"aLH676","date_x":"date","time_x":"time"})
    
    #calculating linear regression to night time measurements
    df_chl_coef = df_chl_night.groupby(["date", "time"])[["chla", "aLH676"]].apply(lambda group: get_linear_reg_coef(group))
    df_chl_coef = df_chl_coef.reset_index()
    df_chl_coef = df_chl_coef.drop_duplicates(["date"], keep="last")
    
    #applying correlation from night time measurements to all measurements
    df_chl = df_chl.merge(df_chl_coef, on=["date"])
    df_chl["chla_corr"] = df_chl["gradient"] * df_chl["aLH676"] + df_chl["intercept"]
    df_chl = df_chl.drop(columns=["time_y"])
    df_chl = df_chl.rename(columns={"time_x":"time"})
    
    df_total = df_total.append(df_chl)
    

C:\Projects\datalakes\thetis-vertical-profiler\thetis-vertical-profiler\notebooks\data_test\Level2\L2_THETIS_GRID_20200131_20200210.nc
        depth                      datetime         par      chla    aLH676  \
12684     1.7 2020-01-31 11:40:53.680999936  382.914065  1.092072  0.015738   
14496     1.8 2020-01-31 11:40:53.680999936  379.751219  1.077664  0.015345   
16308     1.9 2020-01-31 11:40:53.680999936  369.396432  1.091735  0.016355   
18120     2.0 2020-01-31 11:40:53.680999936  363.261179  1.130438  0.016006   
19932     2.1 2020-01-31 11:40:53.680999936  347.543362  1.082286  0.015997   
...       ...                           ...         ...       ...       ...   
881840   49.6 2020-01-31 12:38:22.008000000    0.286593  1.189989  0.131090   
883652   49.7 2020-01-31 12:38:22.008000000    0.278630  1.189989  0.135863   
885464   49.8 2020-01-31 12:38:22.008000000    0.273217  1.186851  0.125274   
887276   49.9 2020-01-31 12:38:22.008000000    0.279891  1.181223  0.115627

ValueError: Found array with 0 sample(s) (shape=(0, 1)) while a minimum of 1 is required.

In [4]:
print(df_total)

      depth                      datetime         par      chla    aLH676  \
0       1.3 2021-09-22 10:59:49.851000064  867.231137  1.207177  0.064195   
1       1.4 2021-09-22 10:59:49.851000064  819.762405  1.205689  0.059439   
2       1.5 2021-09-22 10:59:49.851000064  819.896954  1.231118  0.057426   
3       1.6 2021-09-22 10:59:49.851000064  761.311776  1.203018  0.061343   
4       1.7 2021-09-22 10:59:49.851000064  689.223603  1.177765  0.070975   
...     ...                           ...         ...       ...       ...   
3876   49.7 2022-04-09 10:00:10.847999744    0.104375  0.620411  0.001008   
3877   49.8 2022-04-09 10:00:10.847999744    0.103066  0.622411  0.001428   
3878   49.9 2022-04-09 10:00:10.847999744    0.101769  0.624411  0.001848   
3879   50.0 2022-04-09 10:00:10.847999744    0.100494  0.626411  0.001817   
3880    2.0 2022-04-09 22:00:10.753999872    0.040364  1.443287  0.010215   

            date             time  gradient  intercept  chla_corr  
0     2